In [ ]:
!pip install yfinance -q
import yfinance as yf
import pandas as pd
import requests
import io
import os
from datetime import date

# ==========================================
# SETTINGS
# ==========================================
# Change these anytime! Data won't re-download, only re-calculate.
LOOKBACK_DAYS = 80   # Approx 2000 15m candles
TOP_PCT       = 25   # Premium Zone
BTM_PCT       = 25   # Discount Zone
FORCE_UPDATE  = False # Set True to force re-download intraday

# ==========================================
# CACHING LOGIC
# ==========================================
def get_sp500_tickers():
    filename = 'sp500_tickers_cache.pkl'
    
    if os.path.exists(filename) and not FORCE_UPDATE:
        print("   [CACHE] Loading Tickers from file...")
        return pd.read_pickle(filename)
    
    print("   [WEB] Scraping Wikipedia for Tickers...")
    headers = {"User-Agent": "Mozilla/5.0"}
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    try:
        s = requests.get(url, headers=headers).content
        df = pd.read_html(io.BytesIO(s))[0]
        tickers = df['Symbol'].tolist()
        tickers = [t.replace('.', '-') for t in tickers] # Yahoo Format (BRK-B)
        
        # Save to cache
        pd.to_pickle(tickers, filename)
        return tickers
    except Exception as e:
        print(f"Error: {e}")
        return []

def get_market_data(tickers):
    today_str = date.today().strftime("%Y-%m-%d")
    filename = f'sp500_data_{today_str}.pkl' # File includes date!
    
    if os.path.exists(filename) and not FORCE_UPDATE:
        print(f"   [CACHE] Loading Data from {filename}...")
        return pd.read_pickle(filename)
    
    print("   [WEB] Downloading Daily Data from Yahoo Finance...")
    # Bulk download is fast
    data = yf.download(tickers, period="6mo", interval="1d", progress=True)['Close']
    
    # Save to cache
    data.to_pickle(filename)
    return data

# ==========================================
# MAIN LOGIC
# ==========================================
def run_scanner():
    print("--- STEP 1: TICKERS ---")
    tickers = get_sp500_tickers()
    
    print("\n--- STEP 2: MARKET DATA ---")
    data = get_market_data(tickers)
    
    print("\n--- STEP 3: ANALYZING ZONES ---")
    recent_data = data.tail(LOOKBACK_DAYS)
    
    premium_list = []
    discount_list = []
    
    for ticker in tickers:
        try:
            if ticker not in recent_data.columns: continue
            
            series = recent_data[ticker].dropna()
            if series.empty: continue
            
            curr = series.iloc[-1]
            hi   = series.max()
            lo   = series.min()
            rng  = hi - lo
            
            if rng == 0: continue
            
            loc = (curr - lo) / rng
            
            # Convert Yahoo (BRK-B) to TradingView (BRK.B) format for export
            tv_ticker = ticker.replace('-', '.')
            
            item = {
                'ticker': tv_ticker, 
                'loc': round(loc*100, 1), 
                'price': round(curr, 2)
            }
            
            if loc >= (1.0 - (TOP_PCT/100)):
                premium_list.append(item)
            elif loc <= (BTM_PCT/100):
                discount_list.append(item)
                
        except:
            continue

    # ==========================================
    # OUTPUT TABLES
    # ==========================================
    print("\n" + "="*50)
    print(f"DISCOUNT ZONES (Bottom {BTM_PCT}%) - {len(discount_list)} Stocks")
    print("="*50)
    discount_list.sort(key=lambda x: x['loc'])
    for s in discount_list[:15]: # Show top 15 in console
        print(f"{s['ticker']:<8} {s['loc']:<6}% ${s['price']}")
    if len(discount_list) > 15: print(f"... and {len(discount_list)-15} more.")

    print("\n" + "="*50)
    print(f"PREMIUM ZONES (Top {TOP_PCT}%) - {len(premium_list)} Stocks")
    print("="*50)
    premium_list.sort(key=lambda x: x['loc'], reverse=True)
    for s in premium_list[:15]: # Show top 15 in console
        print(f"{s['ticker']:<8} {s['loc']:<6}% ${s['price']}")
    if len(premium_list) > 15: print(f"... and {len(premium_list)-15} more.")

    # ==========================================
    # TRADINGVIEW IMPORT STRINGS
    # ==========================================
    print("\n\n" + "#"*60)
    print("TRADINGVIEW IMPORT STRINGS (COPY & PASTE)")
    print("#"*60)
    
    print("\n>>> DISCOUNT LIST (Copy below):")
    print(", ".join([x['ticker'] for x in discount_list]))
    
    print("\n>>> PREMIUM LIST (Copy below):")
    print(", ".join([x['ticker'] for x in premium_list]))

run_scanner()